In [1]:
import os
import csv
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import re

# ====== Config ======
MODEL_NAME = "karths/binary_classification_train_TD"
MAX_LENGTH_TOK = 256
SAVE_EVERY = 100

DATASET_NAME = "karths/binary-10IQR-TD"
TEXT_COL = "text"
ID_COL = "id"
LABEL_COL = "label"
N_HEAD = 10_000
N_TAIL = 10_000

OUT_FILE = "risultati_kart-trans-token.csv"
SKIPPED_FILE = "skipped_rows.csv"

# ====== Cleaning ======
_ws_re = re.compile(r"\s+")
def clean_whitespace(text: str) -> str:
    if text is None:
        return ""
    text = str(text).replace("\u00A0", " ")
    text = _ws_re.sub(" ", text)   # collassa spazi/tab/newline
    return text.strip()


# ====== Load teacher ======
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# ====== Load dataset ======
ds = load_dataset(DATASET_NAME)
split = "train" if "train" in ds else list(ds.keys())[0]
data = ds[split]

n = len(data)
head = data.select(range(min(N_HEAD, n)))
tail = data.select(range(max(0, n - N_TAIL), n))

# evita duplicati se n < N_HEAD + N_TAIL
rows = {row[ID_COL]: row for row in list(head) + list(tail)}.values()
rows = list(rows)

# ====== File init (sovrascrive file precedenti automaticamente) ======
def init_csv(path, header):
    f = open(path, "w", newline="", encoding="utf-8")
    writer = csv.DictWriter(
        f,
        fieldnames=header,
        quoting=csv.QUOTE_ALL,  # <-- importantissimo per JSON e testi lunghi
        escapechar="\\"
    )
    writer.writeheader()
    return f, writer

out_f, out_writer = init_csv(
    OUT_FILE,
    ["id", "label", "orig_text", "p_td", "logit_td", "tokens", "token_scores"]
)

skip_f, skip_writer = init_csv(
    SKIPPED_FILE,
    ["id", "reason", "orig_text"]
)

# ====== Helper: token attributions (Grad × Input) ======
def token_attributions_grad_x_input(text: str):
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH_TOK,
        return_tensors="pt"
    ).to(device)

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    embeddings = model.roberta.embeddings.word_embeddings

    model.zero_grad(set_to_none=True)

    inputs_embeds = embeddings(input_ids)
    inputs_embeds.retain_grad()
    inputs_embeds.requires_grad_(True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)

    logits = outputs.logits
    if logits.shape[-1] == 1:
        logit_td = logits.squeeze(-1)
        p_td = torch.sigmoid(logits).squeeze(-1)
    else:
        logit_td = logits[:, 1]
        p_td = torch.softmax(logits, dim=-1)[:, 1]

    logit_td.backward()

    grads = inputs_embeds.grad[0]
    embeds = inputs_embeds.detach()[0]
    token_scores = (grads * embeds).sum(dim=1).abs().detach().cpu().tolist()

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu())

    return tokens, token_scores, float(p_td.item()), float(logit_td.item())

# ====== Main loop ======
buffer = []
processed = 0
skipped = 0
total = len(rows)

print(f"Totale righe da processare: {total} (head={len(head)}, tail={len(tail)}) | device={device}")

for i, row in enumerate(rows):
    frase_id = row.get(ID_COL)
    text = clean_whitespace(row.get(TEXT_COL, None))
    label = row.get(LABEL_COL, None)

    if not text:
        skipped += 1
        skip_writer.writerow({
            "id": frase_id,
            "reason": "empty text",
            "orig_text": ""
        })
        continue

    try:
        tokens, token_scores, p_td, logit_td = token_attributions_grad_x_input(text)
    except Exception as e:
        skipped += 1
        skip_writer.writerow({
            "id": frase_id,
            "reason": f"exception: {type(e).__name__}: {e}",
            "orig_text": text
        })
        continue

    buffer.append({
        "id": frase_id,
        "label": int(label) if label is not None and str(label).strip() != "" else None,
        "orig_text": text,
        "p_td": p_td,
        "logit_td": logit_td,
        "tokens": json.dumps(tokens, ensure_ascii=False),
        "token_scores": json.dumps(token_scores, ensure_ascii=False),
    })

    processed += 1

    if processed % SAVE_EVERY == 0:
        for r in buffer:
            out_writer.writerow(r)
        out_f.flush()
        buffer.clear()

        done = processed + skipped
        pct = (done / total) * 100 if total else 0
        print(f"[checkpoint] ok={processed} skipped={skipped} done={done}/{total} ({pct:.1f}%)")

# flush finale
for r in buffer:
    out_writer.writerow(r)
out_f.flush()

out_f.close()
skip_f.close()

print(f"FINE. ok={processed} skipped={skipped} totale={processed+skipped}")
print("Output:", OUT_FILE)
print("Skipped:", SKIPPED_FILE)


c:\Users\o.sacilotto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 23849.34it/s]


Totale righe da processare: 20000 (head=10000, tail=10000) | device=cpu
[checkpoint] ok=100 skipped=0 done=100/20000 (0.5%)
[checkpoint] ok=200 skipped=0 done=200/20000 (1.0%)
[checkpoint] ok=300 skipped=0 done=300/20000 (1.5%)
[checkpoint] ok=400 skipped=0 done=400/20000 (2.0%)
[checkpoint] ok=500 skipped=0 done=500/20000 (2.5%)
[checkpoint] ok=600 skipped=0 done=600/20000 (3.0%)
[checkpoint] ok=700 skipped=0 done=700/20000 (3.5%)
[checkpoint] ok=800 skipped=0 done=800/20000 (4.0%)
[checkpoint] ok=900 skipped=0 done=900/20000 (4.5%)
[checkpoint] ok=1000 skipped=0 done=1000/20000 (5.0%)
[checkpoint] ok=1100 skipped=0 done=1100/20000 (5.5%)
[checkpoint] ok=1200 skipped=0 done=1200/20000 (6.0%)
[checkpoint] ok=1300 skipped=0 done=1300/20000 (6.5%)
[checkpoint] ok=1400 skipped=0 done=1400/20000 (7.0%)
[checkpoint] ok=1500 skipped=0 done=1500/20000 (7.5%)
[checkpoint] ok=1600 skipped=0 done=1600/20000 (8.0%)
[checkpoint] ok=1700 skipped=0 done=1700/20000 (8.5%)
[checkpoint] ok=1800 skipped